Data Reading

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

PLAYLIST_PATH = "workspace.default.play_list"
INJURY_PATH = "workspace.default.injury_record"
EDA_OUTPUT_PATH = "PlayList_EDA.png"

# Load Data
print("Loading datasets...")
# Read Delta tables directly from Unity Catalog
playlist_delta = spark.read.table(PLAYLIST_PATH)
injury_delta   = spark.read.table(INJURY_PATH)

# Convert to pandas DataFrames for sklearn modeling
playlist = playlist_delta.toPandas()
injury   = injury_delta.toPandas()

print(f"Playlist shape: {playlist.shape}")
print(f"Injury shape:   {injury.shape}")


Data Cleaning

In [0]:
def clean_weather(row):
    cloudy = [
        'Cloudy 50% change of rain', 'Hazy', 'Cloudy.', 'Overcast', 'Mostly Cloudy',
        'Cloudy, fog started developing in 2nd quarter', 'Partly Cloudy',
        'Mostly cloudy', 'Rain Chance 40%', ' Partly cloudy', 'Party Cloudy',
        'Rain likely, temps in low 40s', 'Partly Clouidy', 'Cloudy, 50% change of rain', 'Mostly Coudy', '10% Chance of Rain',
        'Cloudy, chance of rain', '30% Chance of Rain', 'Cloudy, light snow accumulating 1-3"',
        'cloudy', 'Coudy', 'Cloudy with periods of rain, thunder possible. Winds shifting to WNW, 10-20 mph.',
        'Cloudy fog started developing in 2nd quarter', 'Cloudy light snow accumulating 1-3"',
        'Cloudywith periods of rain, thunder possible. Winds shifting to WNW, 10-20 mph.',
        'Cloudy 50% change of rain', 'Cloudy and cold',
        'Cloudy and Cool', 'Partly cloudy'
    ]
    
    clear = [
        'Clear, Windy', ' Clear to Cloudy', 'Clear, highs to upper 80s',
        'Clear and clear', 'Partly sunny',
        'Clear, Windy', 'Clear skies', 'Sunny', 'Partly Sunny', 'Mostly Sunny', 'Clear Skies',
        'Sunny Skies', 'Partly clear', 'Fair', 'Sunny, highs to upper 80s', 'Sun & clouds', 'Mostly sunny', 'Sunny, Windy', 'Mostly Sunny Skies', 'Clear and Sunny', 'Clear and sunny', 'Clear to Partly Cloudy', 'Clear Skies',
        'Clear and cold', 'Clear and warm', 'Clear and Cool', 'Sunny and cold', 'Sunny and warm', 'Sunny and clear'
    ]
    
    rainy = [
        'Rainy', 'Scattered Showers', 'Showers', 'Cloudy Rain', 'Light Rain', 'Rain shower', 'Rain likely, temps in low 40s.', 'Cloudy, Rain'
    ]
    
    snow = [
        'Heavy lake effect snow'
    ]
    
    indoor = [
        'Controlled Climate', 'Indoors', 'N/A Indoor', 'N/A (Indoors)'
    ]
    
    if row.Weather in cloudy:
        return 'Cloudy'
    if row.Weather in indoor:
        return 'Indoor'
    if row.Weather in clear:
        return 'Clear'
    if row.Weather in rainy:
        return 'Rain'
    if row.Weather in snow:
        return 'Snow'
    if row.Weather in ['Cloudy.', 'Heat Index 95', 'Cold']:
        return np.nan
    return row.Weather

def clean_stadiumtype(row):
    if row.StadiumType in ['Bowl', 'Heinz Field', 'Cloudy']:
        return np.nan
    else:
        return row.StadiumType

def clean_play_df(play_df):
    play_df_cleaned = play_df.copy()
    
    # clean StadiumType
    play_df_cleaned['StadiumType'] = play_df_cleaned['StadiumType'].str.replace(
        r'Oudoor|Outdoors|Ourdoor|Outddors|Outdor|Outside', 'Outdoor'
    )
    play_df_cleaned['StadiumType'] = play_df_cleaned['StadiumType'].str.replace(
        r'Indoors|Indoor, Roof Closed|Indoor, Open Roof', 'Indoor'
    )
    play_df_cleaned['StadiumType'] = play_df_cleaned['StadiumType'].str.replace(
        r'Closed Dome|Domed, closed|Domed, Open|Domed, open|Dome, closed|Domed', 'Dome'
    )
    play_df_cleaned['StadiumType'] = play_df_cleaned['StadiumType'].str.replace(
        r'Retr. Roof-Closed|Outdoor Retr Roof-Open|Retr. Roof - Closed|Retr. Roof-Open|Retr. Roof - Open|Retr. Roof Closed', 'Retractable Roof'
    )
    play_df_cleaned['StadiumType'] = play_df_cleaned['StadiumType'].str.replace(
        'Open', 'Outdoor'
    )
    play_df_cleaned['StadiumType'] = play_df_cleaned.apply(
        lambda row: clean_stadiumtype(row), axis=1
    )
    
    # clean Weather
    play_df_cleaned['Weather'] = play_df_cleaned.apply(
        lambda row: clean_weather(row), axis=1
    )
    
    # Save cleaned DataFrame to CSV
    play_df_cleaned.to_csv("play_list_cleaned.csv", index=False)
    
    return play_df_cleaned

In [0]:
print("\nCleaning data..")
injury["Injured"] = 1

clean_df = playlist.merge(
    injury[["PlayKey", "Injured", "BodyPart", "Surface"]],
    on="PlayKey",
    how="left"
)
clean_df["Injured"] = clean_df["Injured"].fillna(0).astype(int)

print(f"Initial merged dataset shape: {clean_df.shape}")
print(f"Total injuries in data: {clean_df['Injured'].sum()}")

clean_df = clean_df.dropna(subset=[
    "PositionGroup", "Weather", "Temperature",
    "FieldType", "StadiumType"
])
clean_df = clean_df[clean_df["PositionGroup"] != "Missing Data"]
clean_df = clean_df[clean_df["Temperature"] > -100]

# Normalize StadiumType
def categorize_stadium(s):
    s = str(s).lower()
    if any(x in s for x in ['indoor', 'dome', 'closed', 'retractable']):
        return 'Indoor/Dome'
    elif any(x in s for x in ['outdoor', 'open', 'bowl', 'outside', 'heinz',
                               'ourdoor', 'outdor', 'oudoor', 'outddors']):
        return 'Outdoor'
    else:
        return 'Other'

clean_df['StadiumCat'] = clean_df['StadiumType'].apply(categorize_stadium)

position_counts = clean_df.groupby("PositionGroup")['Injured'].sum()
print("\nInjury counts by position:")
for position, count in position_counts.items():
    print(f"  {position}: {int(count)} injuries")

print(f"\nFiltered dataset shape: {clean_df.shape}")

EDA

In [0]:
print("\nGenerating EDA dashboard...")

BLUE   = '#3B82F6'
GREEN  = '#10B981'
ORANGE = '#F59E0B'
RED    = '#EF4444'
PURPLE = '#8B5CF6'
GRAY   = '#6B7280'
PALETTE = [BLUE, GREEN, ORANGE, RED, PURPLE, GRAY,
           '#EC4899', '#14B8A6', '#F97316', '#84CC16']

fig = plt.figure(figsize=(22, 26), facecolor='#F8FAFC')
fig.suptitle('NFL Play Dataset — Exploratory Data Analysis',
             fontsize=22, fontweight='bold', color='#1E293B', y=0.98)

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Play Type Distribution
ax1 = fig.add_subplot(gs[0, 0])
pt = clean_df['PlayType'].value_counts().drop('0', errors='ignore').head(8)
bars = ax1.barh(pt.index[::-1], pt.values[::-1],
                color=PALETTE[:len(pt)], edgecolor='white', linewidth=0.5)
ax1.set_title('Play Type Distribution', fontweight='bold', color='#1E293B', pad=10)
ax1.set_xlabel('Count', color='#475569')
ax1.tick_params(colors='#475569', labelsize=8)
for bar, val in zip(bars, pt.values[::-1]):
    ax1.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=7.5, color='#475569')
ax1.spines[['top', 'right']].set_visible(False)
ax1.set_facecolor('#F8FAFC')

# 2. Roster Position Split
ax2 = fig.add_subplot(gs[0, 1])
pg = clean_df['RosterPosition'].value_counts()
_, _, autotexts = ax2.pie(pg.values, labels=None, autopct='%1.1f%%',
                           colors=PALETTE[:len(pg)], startangle=140,
                           pctdistance=0.75,
                           wedgeprops=dict(edgecolor='white', linewidth=1.2))
for at in autotexts:
    at.set_fontsize(7)
ax2.legend(pg.index, loc='lower left', fontsize=6.5,
           bbox_to_anchor=(-0.15, -0.15), ncol=2)
ax2.set_title('Roster Position Split', fontweight='bold', color='#1E293B', pad=10)

# 3. Field Type
ax3 = fig.add_subplot(gs[0, 2])
ft = clean_df['FieldType'].value_counts()
bars3 = ax3.bar(ft.index, ft.values, color=[GREEN, ORANGE],
                edgecolor='white', linewidth=0.8, width=0.5)
ax3.set_title('Field Type', fontweight='bold', color='#1E293B', pad=10)
ax3.set_ylabel('Plays', color='#475569')
for bar, val in zip(bars3, ft.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 800,
             f'{val:,}', ha='center', fontsize=10, fontweight='bold', color='#1E293B')
ax3.spines[['top', 'right']].set_visible(False)
ax3.tick_params(colors='#475569')
ax3.set_facecolor('#F8FAFC')

# 4. Temperature Distribution
ax4 = fig.add_subplot(gs[1, 0])
temps = clean_df['Temperature']
ax4.hist(temps, bins=40, color=BLUE, edgecolor='white', linewidth=0.4, alpha=0.85)
ax4.axvline(temps.median(), color=RED, linestyle='--', linewidth=1.5,
            label=f'Median: {temps.median():.0f}°F')
ax4.set_title('Temperature Distribution', fontweight='bold', color='#1E293B', pad=10)
ax4.set_xlabel('Temperature (°F)', color='#475569')
ax4.set_ylabel('Plays', color='#475569')
ax4.legend(fontsize=8)
ax4.spines[['top', 'right']].set_visible(False)
ax4.tick_params(colors='#475569')
ax4.set_facecolor('#F8FAFC')

# 5. Plays per Player per Game
ax5 = fig.add_subplot(gs[1, 1])
plays_per_game = clean_df.groupby(['PlayerKey', 'PlayerGame']).size().reset_index(name='plays')
ax5.hist(plays_per_game['plays'], bins=30, color=PURPLE,
         edgecolor='white', linewidth=0.4, alpha=0.85)
ax5.set_title('Plays per Player per Game', fontweight='bold', color='#1E293B', pad=10)
ax5.set_xlabel('Number of Plays', color='#475569')
ax5.set_ylabel('Frequency', color='#475569')
med = plays_per_game['plays'].median()
ax5.axvline(med, color=ORANGE, linestyle='--', linewidth=1.5, label=f'Median: {med:.0f}')
ax5.legend(fontsize=8)
ax5.spines[['top', 'right']].set_visible(False)
ax5.tick_params(colors='#475569')
ax5.set_facecolor('#F8FAFC')

# 6. Stadium Type (Normalized)
ax6 = fig.add_subplot(gs[1, 2])
sc = clean_df['StadiumCat'].value_counts()
_, _, autotexts6 = ax6.pie(sc.values, labels=sc.index, autopct='%1.1f%%',
                            colors=[BLUE, GREEN, GRAY], startangle=90,
                            wedgeprops=dict(edgecolor='white', linewidth=1.5))
for at in autotexts6:
    at.set_fontsize(9)
ax6.set_title('Stadium Type (Normalized)', fontweight='bold', color='#1E293B', pad=10)

# 7. Play Type by Field Type
ax7 = fig.add_subplot(gs[2, :2])
top_plays = ['Pass', 'Rush', 'Extra Point', 'Kickoff', 'Punt', 'Field Goal']
pt_ft = (clean_df[clean_df['PlayType'].isin(top_plays)]
         .groupby(['PlayType', 'FieldType']).size().unstack(fill_value=0))
x = np.arange(len(pt_ft))
w = 0.35
ax7.bar(x - w/2, pt_ft.get('Natural',   0), width=w, label='Natural',   color=GREEN,  edgecolor='white')
ax7.bar(x + w/2, pt_ft.get('Synthetic', 0), width=w, label='Synthetic', color=ORANGE, edgecolor='white')
ax7.set_xticks(x)
ax7.set_xticklabels(pt_ft.index, rotation=20, ha='right', color='#475569')
ax7.set_title('Play Type by Field Type', fontweight='bold', color='#1E293B', pad=10)
ax7.set_ylabel('Plays', color='#475569')
ax7.legend()
ax7.spines[['top', 'right']].set_visible(False)
ax7.tick_params(colors='#475569')
ax7.set_facecolor('#F8FAFC')

# 8. Temperature by Stadium Type
ax8 = fig.add_subplot(gs[2, 2])
cats = ['Outdoor', 'Indoor/Dome']
data_box = [clean_df[clean_df['StadiumCat'] == c]['Temperature'].dropna().values for c in cats]
bp = ax8.boxplot(data_box, labels=cats, patch_artist=True,
                 medianprops=dict(color=RED, linewidth=2))
for patch, color in zip(bp['boxes'], [BLUE, GREEN]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax8.set_title('Temperature by Stadium Type', fontweight='bold', color='#1E293B', pad=10)
ax8.set_ylabel('Temperature (°F)', color='#475569')
ax8.tick_params(colors='#475569')
ax8.spines[['top', 'right']].set_visible(False)
ax8.set_facecolor('#F8FAFC')

# 9. Top 10 Positions by Play Volume
ax9 = fig.add_subplot(gs[3, 0])
top_pos = clean_df['Position'].value_counts().head(10)
ax9.bar(top_pos.index, top_pos.values, color=PALETTE[:len(top_pos)], edgecolor='white')
ax9.set_title('Top 10 Positions by Play Volume', fontweight='bold', color='#1E293B', pad=10)
ax9.set_ylabel('Plays', color='#475569')
ax9.tick_params(axis='x', rotation=45, colors='#475569', labelsize=8)
ax9.tick_params(axis='y', colors='#475569')
ax9.spines[['top', 'right']].set_visible(False)
ax9.set_facecolor('#F8FAFC')

# 10. Injury Rate by Position Group
ax10 = fig.add_subplot(gs[3, 1])
inj_by_pos = (clean_df.groupby('PositionGroup')['Injured']
              .mean().sort_values(ascending=False) * 100)
ax10.barh(inj_by_pos.index[::-1], inj_by_pos.values[::-1],
          color=RED, edgecolor='white', alpha=0.8)
ax10.set_title('Injury Rate by Position Group (%)', fontweight='bold', color='#1E293B', pad=10)
ax10.set_xlabel('Injury Rate (%)', color='#475569')
ax10.spines[['top', 'right']].set_visible(False)
ax10.tick_params(colors='#475569', labelsize=8)
ax10.set_facecolor('#F8FAFC')

# 11. Injury Rate by Field Type
ax11 = fig.add_subplot(gs[3, 2])
inj_by_field = (clean_df.groupby('FieldType')['Injured']
                .mean().sort_values(ascending=False) * 100)
bars11 = ax11.bar(inj_by_field.index, inj_by_field.values,
                  color=[GREEN, ORANGE], edgecolor='white', width=0.5)
ax11.set_title('Injury Rate by Field Type (%)', fontweight='bold', color='#1E293B', pad=10)
ax11.set_ylabel('Injury Rate (%)', color='#475569')
for bar, val in zip(bars11, inj_by_field.values):
    ax11.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
              f'{val:.3f}%', ha='center', fontsize=9, fontweight='bold', color='#1E293B')
ax11.spines[['top', 'right']].set_visible(False)
ax11.tick_params(colors='#475569')
ax11.set_facecolor('#F8FAFC')

plt.savefig(EDA_OUTPUT_PATH, dpi=150, bbox_inches='tight', facecolor='#F8FAFC')
plt.show()
print(f"EDA chart saved → {EDA_OUTPUT_PATH}")

Feature Engineering

In [0]:
features = ["PositionGroup", "Weather", "Temperature", "FieldType", "StadiumType"]
X = clean_df[features]
y = clean_df["Injured"]

categorical_features = ["PositionGroup", "Weather", "FieldType", "StadiumType"]
numerical_features   = ["Temperature"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

model = Pipeline([
    ("preprocess",  preprocessor),
    ("classifier",  RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

Correlation Analysis


In [0]:
import seaborn as sns

# One-hot encode Weather
weather_corr_df = pd.get_dummies(clean_df["Weather"], drop_first=False)
weather_corr_df["Injured"] = clean_df["Injured"]

# Compute correlation matrix
corr_matrix = weather_corr_df.corr()

# Plot heatmap for Weather columns and Injury
plt.figure(figsize=(10, 6))
sns.heatmap(
    corr_matrix,
    annot=True, cmap="coolwarm", cbar=True, linewidths=0.5
)
plt.title("Correlation Heatmap: Weather vs Injury")
plt.ylabel("Features")
plt.xlabel("Features")
plt.tight_layout()
plt.show()

Train Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

In [0]:
from sklearn.utils import resample

# Combine X_train and y_train for upsampling
train_df = X_train.copy()
train_df['Injured'] = y_train.values

# Separate majority and minority classes
majority = train_df[train_df['Injured'] == 0]
minority = train_df[train_df['Injured'] == 1]

print(f"Minority class samples (injuries): {len(minority)}")
print(f"Majority class samples (no injuries): {len(majority)}")

# Upsample minority (injury) cases
minority_upsampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

# Combine majority with upsampled minority
upsampled_train_df = pd.concat([majority, minority_upsampled])

# Separate features and labels
X_train_upsampled = upsampled_train_df.drop('Injured', axis=1)
y_train_upsampled = upsampled_train_df['Injured']

print(f"New injury count after oversampling: {y_train_upsampled.sum()} out of {len(y_train_upsampled)}")
print(f"Class distribution after oversampling:\n{y_train_upsampled.value_counts()}")

In [0]:
%pip install xgboost

Model Training

In [0]:
print("\nTraining model (with oversampled data)...")
model.fit(X_train_upsampled, y_train_upsampled)

Model Evalutation

In [0]:
print("MODEL EVALUATION (Oversampled Training Set)")

print("\nTRAINING SET EVALUATION (Oversampled):")
y_train_pred = model.predict(X_train_upsampled)
train_accuracy = accuracy_score(y_train_upsampled, y_train_pred)
print(f"Training Accuracy: {train_accuracy:.3f}")
print(confusion_matrix(y_train_upsampled, y_train_pred))
print(classification_report(y_train_upsampled, y_train_pred))

In [0]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC

# Logistic Regression model
logreg_model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])
logreg_model.fit(X_train_upsampled, y_train_upsampled)

# XGBoost model
xgb_model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=100,
        max_depth=10,
        scale_pos_weight=1,
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"
    ))
])
xgb_model.fit(X_train_upsampled, y_train_upsampled)



In [0]:
print("XGBOOST CLASSIFIER RESULTS")
y_pred_xgb = xgb_model.predict(X_train_upsampled)
xgb_accuracy = accuracy_score(y_train_upsampled, y_pred_xgb)
print(f"\nXGBoost Accuracy: {xgb_accuracy:.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_train_upsampled, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_train_upsampled, y_pred_xgb))

print("\nLOGISTIC REGRESSION RESULTS")
y_pred_logreg = logreg_model.predict(X_train_upsampled)
logreg_accuracy = accuracy_score(y_train_upsampled, y_pred_logreg)
print(f"\nLogistic Regression Accuracy: {logreg_accuracy:.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_train_upsampled, y_pred_logreg))
print("\nClassification Report:")
print(classification_report(y_train_upsampled, y_pred_logreg))



In [0]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, make_scorer
import os

# Define parameter grids (reduced for computational efficiency)
rf_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 12],
    'classifier__class_weight': ['balanced']
}
logreg_param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__penalty': ['l2'],
    'classifier__class_weight': ['balanced']
}
xgb_param_grid = {
    'classifier__n_estimators': [100],
    'classifier__max_depth': [10, 12],
    'classifier__scale_pos_weight': [1, 2]
}

# Set up cross-validation and scoring (reduced folds from 5 to 3)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score),
    'roc_auc': make_scorer(roc_auc_score)
}

# Run GridSearchCV for each model
def run_gridsearch(model, param_grid, X, y, model_name):
    print(f"\nStarting GridSearchCV for {model_name}...")
    grid = GridSearchCV(
        model, param_grid, cv=cv, scoring=scoring, refit='f1', n_jobs=-1, 
        return_train_score=False, verbose=1
    )
    grid.fit(X, y)
    results = pd.DataFrame(grid.cv_results_)
    results.to_csv(f"{model_name}_gridsearch_results.csv", index=False)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X)
    y_proba = best_model.predict_proba(X)[:, 1] if hasattr(best_model, "predict_proba") else None
    metrics = {
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'F1-score': f1_score(y, y_pred),
        'Confusion Matrix': confusion_matrix(y, y_pred),
        'ROC-AUC': roc_auc_score(y, y_proba) if y_proba is not None else None
    }
    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(f"{model_name}_metrics.csv", index=False)
    print(f"{model_name} complete. Best params: {grid.best_params_}")
    return grid.best_params_, metrics

# Run for each model
rf_best_params, rf_metrics = run_gridsearch(model, rf_param_grid, X_train_upsampled, y_train_upsampled, "RandomForest")
logreg_best_params, logreg_metrics = run_gridsearch(logreg_model, logreg_param_grid, X_train_upsampled, y_train_upsampled, "LogisticRegression")
xgb_best_params, xgb_metrics = run_gridsearch(xgb_model, xgb_param_grid, X_train_upsampled, y_train_upsampled, "XGBoost")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Save upsampled training dataset
upsampled_train_df.to_csv("cleaned_dataset.csv", index=False)

# Save trained models
joblib.dump(model, "RandomForest_model.pkl")
joblib.dump(logreg_model, "LogisticRegression_model.pkl")
joblib.dump(xgb_model, "XGBoost_model.pkl")

# Create balanced test set: 50 injured, 50 non-injured plays
injured_plays = clean_df[clean_df["Injured"] == 1].sample(n=50, random_state=42)
noninjured_plays = clean_df[clean_df["Injured"] == 0].sample(n=50, random_state=42)
balanced_test_df = pd.concat([injured_plays, noninjured_plays]).sample(frac=1, random_state=42)  # shuffle

X_balanced_test = balanced_test_df[features]
y_balanced_test = balanced_test_df["Injured"]

# Model evaluation metrics (using balanced test set)
models = {
    "RandomForest": model,
    "LogisticRegression": logreg_model,
    "XGBoost": xgb_model
}
metrics_list = []
conf_matrices = {}

for name, mdl in models.items():
    y_pred = mdl.predict(X_balanced_test)
    accuracy = accuracy_score(y_balanced_test, y_pred)
    precision = precision_score(y_balanced_test, y_pred)
    recall = recall_score(y_balanced_test, y_pred)
    f1 = f1_score(y_balanced_test, y_pred)
    conf_matrix = confusion_matrix(y_balanced_test, y_pred)
    conf_matrices[name] = conf_matrix
    metrics_list.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Notes": "",
        "Confusion Matrix": conf_matrix.tolist()
    })

# Identify best model based on F1 Score and Recall
comparison_df = pd.DataFrame(metrics_list)
best_f1_idx = comparison_df['F1 Score'].idxmax()
best_recall_idx = comparison_df['Recall'].idxmax()
best_f1_model = comparison_df.loc[best_f1_idx, 'Model']
best_recall_model = comparison_df.loc[best_recall_idx, 'Model']

for i, row in comparison_df.iterrows():
    notes = []
    if row['Model'] == best_f1_model:
        notes.append("Best F1 Score")
    if row['Model'] == best_recall_model:
        notes.append("Best Recall")
    comparison_df.at[i, "Notes"] = ", ".join(notes) if notes else ""

display(comparison_df)

# Save combined results to CSV
comparison_df.to_csv("model_comparison_balanced_test.csv", index=False)

# Confusion matrix visualization
plt.figure(figsize=(18, 5))
for i, (name, matrix) in enumerate(conf_matrices.items()):
    plt.subplot(1, 3, i+1)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Documentation of hyperparameter tuning results
rf_gridsearch_results = pd.read_csv("RandomForest_gridsearch_results.csv")
logreg_gridsearch_results = pd.read_csv("LogisticRegression_gridsearch_results.csv")
xgb_gridsearch_results = pd.read_csv("XGBoost_gridsearch_results.csv")

display(rf_gridsearch_results.head())
display(logreg_gridsearch_results.head())
display(xgb_gridsearch_results.head())

# Final model comparison discussion placeholder
print("\nMODEL COMPARISON DISCUSSION:")
print("See comparison table and confusion matrices above. Hyperparameter tuning results are documented in the CSV files. The best performing model can be selected based on F1 Score and recall, depending on the importance of injury detection.")

In [0]:
print("MODEL EVALUATION")

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.3f}")

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred))

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

Validation Test set, create a random test set with 50 injured and non injured.
Test hyperparameters on RandomForest 
Split 50 injured observations into two datasets, use one for oversampling

Feature Importance

In [0]:
print("TOP 10 FEATURE IMPORTANCE (Excluding PositionGroup, Training Set Only)")

# Recombine similar Weather and StadiumType values for feature importance display
def combine_feature_names(feature_names):
    replacements = {
        "Weather_Mostly Cloudy": "Weather_Cloudy",
        "Weather_Cloudy.": "Weather_Cloudy",
        "Weather_Cloudy, 50% change of rain": "Weather_Cloudy",
        "Weather_Mostly Coudy": "Weather_Cloudy",
        "Weather_Cloudy, chance of rain": "Weather_Cloudy",
        "Weather_Cloudy, fog started developing in 2nd quarter": "Weather_Cloudy",
        "Weather_Cloudywith periods of rain, thunder possible. Winds shifting to WNW, 10-20 mph.": "Weather_Cloudy",
        "Weather_Cloudy light snow accumulating 1-3\"": "Weather_Cloudy",
        "Weather_Cloudy fog started developing in 2nd quarter": "Weather_Cloudy",
        "Weather_Cloudy and cold": "Weather_Cloudy",
        "Weather_Cloudy and Cool": "Weather_Cloudy",
        "Weather_Mostly Coudy": "Weather_Cloudy",
        "StadiumType_Outdoors": "StadiumType_Outdoor",
        "StadiumType_Outddors": "StadiumType_Outdoor",
        "StadiumType_Ourdoor": "StadiumType_Outdoor",
        "StadiumType_Outdor": "StadiumType_Outdoor",
        "StadiumType_Oudoor": "StadiumType_Outdoor",
    }
    return [replacements.get(f, f) for f in feature_names]

model.fit(X_train, y_train)
feature_names = (
    list(model.named_steps['preprocess']
         .named_transformers_['cat']
         .get_feature_names_out(categorical_features)) +
    numerical_features
)
combined_feature_names = combine_feature_names(feature_names)
rf_importance_df = pd.DataFrame({
    'Feature':    combined_feature_names,
    'Importance': model.named_steps['classifier'].feature_importances_
})
rf_importance_df = (rf_importance_df[~rf_importance_df["Feature"].str.contains("PositionGroup")]
                    .groupby("Feature", as_index=False)
                    .agg({"Importance": "sum"})
                    .sort_values('Importance', ascending=False))
print("\nTop 10 Most Important Features (Random Forest, Training Set):")
print(rf_importance_df.head(10).to_string(index=False))

# XGBoost feature importance
xgb_model.fit(X_train, y_train)
xgb_importance_df = pd.DataFrame({
    'Feature':    combine_feature_names(feature_names),
    'Importance': xgb_model.named_steps['classifier'].feature_importances_
})
xgb_importance_df = (xgb_importance_df[~xgb_importance_df["Feature"].str.contains("PositionGroup")]
                     .groupby("Feature", as_index=False)
                     .agg({"Importance": "sum"})
                     .sort_values('Importance', ascending=False))
print("\nTop 10 Most Important Features (XGBoost, Training Set):")
print(xgb_importance_df.head(10).to_string(index=False))

# Logistic Regression feature importance (absolute value of coefficients)
logreg_model.fit(X_train, y_train)
logreg_coef = logreg_model.named_steps['classifier'].coef_[0]
logreg_importance_df = pd.DataFrame({
    'Feature':    combine_feature_names(feature_names),
    'Importance': abs(logreg_coef)
})
logreg_importance_df = (logreg_importance_df[~logreg_importance_df["Feature"].str.contains("PositionGroup")]
                        .groupby("Feature", as_index=False)
                        .agg({"Importance": "sum"})
                        .sort_values('Importance', ascending=False))
print("\nTop 10 Most Important Features (Logistic Regression, Training Set):")
print(logreg_importance_df.head(10).to_string(index=False))

Injury Probability Prediction

In [0]:
clean_df["Injury_Probability"] = model.predict_proba(X)[:, 1]


Injury Risk Summary by Position Group

In [0]:
print("INJURY RISK BY POSITION GROUP")

position_summary = (
    clean_df.groupby("PositionGroup")
    .agg(
        Avg_Injury_Probability=("Injury_Probability", "mean"),
        Injury_Count=("Injured", "sum"),
        Total_Plays=("Injured", "count")
    )
    .reset_index()
    .sort_values("Avg_Injury_Probability", ascending=False)
)
print(position_summary.to_string(index=False))

Injury Risk Summary by Field Type

In [0]:
print("INJURY RISK BY FIELD TYPE")

field_summary = (
    clean_df.groupby("FieldType")
    .agg(
        Avg_Injury_Probability=("Injury_Probability", "mean"),
        Injury_Count=("Injured", "sum"),
        Total_Plays=("Injured", "count")
    )
    .reset_index()
    .sort_values("Avg_Injury_Probability", ascending=False)
)
print(field_summary.to_string(index=False))

Summary Statistics

In [0]:
print("SUMMARY STATISTICS")
print(f"Total plays analyzed: {len(clean_df)}")
print(f"Total injuries:       {clean_df['Injured'].sum()}")
print(f"Positions included:   {list(position_summary['PositionGroup'].unique())}")

print("\nAnalysis complete!")